# 🎬 LivePortrait + Wav2Lip: Free High-Fidelity HR Avatar Generator

This notebook provides a 100% free pipeline to generate hyper-realistic talking-head video clips for your AI HR recruiters using the free T4 GPU on Google Colab.

### 💡 How the Hybrid Pipeline Works:
1. **LivePortrait:** Animates your static recruiter portrait using a generic driving video (adding natural head nodding, eye blinking, and shoulder movement) to eliminate stiffness.
2. **Wav2Lip:** Syncs the lips of the animated recruiter video to the generated audio track.
3. **Result:** An ultra-realistic video where the recruiter looks alive, moves naturally, and speaks the exact text.

## 🚀 Step 1: Install LivePortrait & Wav2Lip Environments

In [ ]:
# Clone LivePortrait repository
!git clone https://github.com/KwaiVGI/LivePortrait.git
%cd LivePortrait

# Install requirements
!pip install -r requirements.txt

# Download pre-trained weights
!mkdir -p pretrained_weights
!wget https://huggingface.co/KwaiVGI/LivePortrait/resolve/main/all_models.zip -O pretrained_weights/all_models.zip
!unzip -d pretrained_weights/ pretrained_weights/all_models.zip

print("✓ LivePortrait Setup Complete!")

## 🎙️ Step 2: Clone and Setup Wav2Lip

In [ ]:
%cd /content
# Clone Wav2Lip repository
!git clone https://github.com/Rudrabha/Wav2Lip.git
%cd Wav2Lip

# Install dependencies
!pip install gdown
!pip install librosa==0.9.1

# Download Wav2Lip pre-trained models
!mkdir -p checkpoints
!wget "https://iiitaphyd-my.sharepoint.com/:u:/g/personal/radrabha_m_research_iiit_ac_in/Eb3LEzZGCn1Hs1aTaJWDyQIBOA2wQD7tgnfE3X4Z14aTaA?download=1" -O checkpoints/wav2lip_gan.pth

print("✓ Wav2Lip Setup Complete!")

## 🎬 Step 3: Run the Pipeline

Upload your static recruiter portrait (e.g. `sarah_chen.png`) and your audio file (e.g. `hello.wav`) to `/content/`, then run the following cell:

In [ ]:
import os
import subprocess

# Define your input files
portrait_path = "/content/sarah_chen.png"
audio_path = "/content/hello.wav"

assert os.path.exists(portrait_path), "Please upload your portrait image!"
assert os.path.exists(audio_path), "Please upload your audio file!"

print("1. Animating portrait with LivePortrait (node/blink movement)...")
%cd /content/LivePortrait

# Run LivePortrait inference using default driving video template
cmd_lp = [
    "python", "inference.py",
    "--source", portrait_path,
    "--driving", "assets/examples/driving/d0.mp4",
    "--output-dir", "/content/liveportrait_output",
    "--no-flag-lip"  # Let Wav2Lip handle mouth movements
]
subprocess.run(cmd_lp, check=True)

# Retrieve the animated video filename
lp_video_dir = "/content/liveportrait_output"
generated_files = [f for f in os.listdir(lp_video_dir) if f.endswith('.mp4')]
assert len(generated_files) > 0, "LivePortrait failed to generate video!"
animated_video_path = os.path.join(lp_video_dir, generated_files[0])

print("2. Synchronizing mouth/lips using Wav2Lip...")
%cd /content/Wav2Lip

output_video_path = "/content/final_talking_recruiter.mp4"
cmd_w2l = [
    "python", "inference.py",
    "--checkpoint_path", "checkpoints/wav2lip_gan.pth",
    "--face", animated_video_path,
    "--audio", audio_path,
    "--outfile", output_video_path
]
subprocess.run(cmd_w2l, check=True)

print(f"✓ Process finished! Download your final video from: {output_video_path}")